In [1]:
.libPaths(c('/home/jbos/.conda/envs/vcfR','/home/jbos/.conda/envs/vcfR/lib/R/library','/software/apps/r/conda3-4.4.1/lib/R/library','/home/jbos/R/x86_64-conda-linux-gnu-library/4.4'))

In [3]:
#Load required packages
library(adegenet)
library(hierfstat)
library(tidyverse)
library(vegan)
library(vcfR)
library(sp)
library(sf)

In [4]:
#Calculate great circle distances from lat/lons
wide_dists <- function(sites){
    x<-st_as_sf(sites,coords=c('LONGITUDE','LATITUDE'))
    x<-st_set_crs(x,4326)
    d <- st_distance(x)
    colnames(d)<-sites$Name
    rownames(d)<-sites$Name
    d<-as.data.frame(d)
    return(d)
}

In [5]:
#Calculate great circle distances from lat/lons
wide_dists2 <- function(sites){
    x<-st_as_sf(sites,coords=c('lon_flat','lat_flat'))
    x<-st_set_crs(x,4326)
    d <- st_distance(x)
    colnames(d)<-sites$release_flat
    rownames(d)<-sites$release_flat
    d<-as.data.frame(d)
    return(d)
}

In [6]:
#Inputs: wide form FSTs and dataframe of individuals corresponding to each population
#Output: long form FSTs. Columns: siteA	siteB	fst	siteA_counts	siteB_counts	mean_counts
fst_transform<-function(fsts,indpops){
    fst_long<-pivot_longer(fsts,cols=colnames(fsts)[2:length(colnames(fsts))],names_to='siteB',values_to='fst')
    colnames(fst_long)<-c('siteA','siteB','fst')
    fst_long<-fst_long[!is.nan(fst_long$fst),]
    dum<-as.data.frame(table(indpops$Population))
    colnames(dum)<-c('siteA','siteA_counts')
    fst_long<-left_join(fst_long,dum,by='siteA')
    colnames(dum)<-c('siteB','siteB_counts')
    fst_long<-left_join(fst_long,dum,by='siteB')
    fst_long$mean_counts<-rowMeans(as.data.frame(cbind(fst_long$siteA_counts,fst_long$siteB_counts)))
    return(fst_long)
}

In [7]:
#Inputs: wide form FSTs and dataframe of individuals corresponding to each population
#Output: wide form linearized FSTs for Mantel tests and figures
linear_fsts<-function(wide_fsts){
  wide_fsts<-as.data.frame(wide_fsts)
  rownames(wide_fsts)<-wide_fsts[,1]
  wide_fsts<-wide_fsts[,-1]
  fst_lin <- wide_fsts/(1-wide_fsts)
  return(fst_lin)
}

In [8]:
#Inputs: wide form FSTs and dataframe of individuals corresponding to each population
#Output: wide form linearized FSTs for Mantel tests and figures
linear_fsts2<-function(wide_fsts){
  wide_fsts<-as.data.frame(wide_fsts)
  fst_lin <- wide_fsts/(1-wide_fsts)
  return(fst_lin)
}

In [9]:
#Inputs: wide form linearized FSTs and wide form distances for each cryptic taxon
#Outputs: long form dataframe with columns SiteA	SiteB	Lin_FST	Distance_m
fst_dist_join<-function(fsts_linear, distances){
    fsts_linear$SiteA<-rownames(fsts_linear)
    distances$SiteA<-rownames(distances)
    fst_lin_lon<-pivot_longer(data=fsts_linear,cols=colnames(fsts_linear)[!(colnames(fsts_linear) %in% 'SiteA')],names_to='SiteB',values_to='Lin_FST')
    dists_lon<-pivot_longer(data=distances,cols=colnames(distances)[!(colnames(distances) %in% 'SiteA')],names_to='SiteB',values_to='Distance_m')
    mer<-inner_join(fst_lin_lon,dists_lon)
    mer<-mer[mer$SiteA!=mer$SiteB,]
    mer$Distance_m<-as.numeric(mer$Distance_m)
    mer_uniquerows <- mer %>% distinct(Distance_m, .keep_all = TRUE)
    return(mer_uniquerows)
    }

In [10]:
#Count individuals per population
make_counts<-function(inds_per_pop){
    dummy<-as.data.frame(matrix(nrow=length(unique(inds_per_pop$Population))))
    dummy[,1]<-unique(inds_per_pop$Population)
    dummy[,2]<-table(inds_per_pop$Population)
    colnames(dummy)<-c('pop','count')
    return(dummy)
    }

In [11]:
#Function to create long form dataset containing pairwise FSTs and corresponding release sites.
#Note that due to differing spatial grains multiple pairwise FST values will correspond to each release site pair!
fst_transform2<-function(fsts,sitesA,sitesB){
    colnames(fsts)[1]<-'siteA'
    fst_long<-pivot_longer(fsts,cols=colnames(fsts)[2:length(colnames(fsts))],names_to='siteB',values_to='fst')
    colnames(fst_long)<-c('siteA','siteB','fst')
    fst_long<-fst_long[!is.nan(fst_long$fst),]
    fst_long<-left_join(fst_long,sitesA,by='siteA')
    fst_long<-left_join(fst_long,sitesB,by='siteB')
    fst_long$release<-paste(fst_long$release_siteA,fst_long$release_siteB)
    return(fst_long)
}

In [12]:
#Stick together data for figure making
join_for_fig<-function(merged_data,long_fsts){
    merged_data$SiteAB<-paste(merged_data$SiteA,merged_data$SiteB)
    long_fsts$SiteAB<-paste(long_fsts$siteA,long_fsts$siteB)
    new<-left_join(merged_data,long_fsts,by='SiteAB')
    new$Distance_km<-new$Distance_m/1000
    new<-new[,c('SiteA','SiteB','Lin_FST','Distance_m','mean_counts','Distance_km')]
    }

In [13]:
#Input: FSTs, vector of number of individuals per population, minimum number of individuals per popualtion
#Output: Matrix of FSTs including only populations above the minimum size
fst_trim_by_N<-function(fsts,inds_per_pop,min_n){
    fsts<-as.data.frame(fsts)
    fst_trim<-fsts[rownames(fsts) %in% as.data.frame(table(inds_per_pop$Population)[table(inds_per_pop$Population)>(min_n-1)])$Var1,]
    fst_trim<-fst_trim[,colnames(fst_trim)%in% as.data.frame(table(inds_per_pop$Population)[table(inds_per_pop$Population)>(min_n-1)])$Var1]
    rownames(fst_trim)<-colnames(fst_trim)
    return(fst_trim)
    }

In [14]:
#Import site locations with lat/lons
sites <- read.csv('/home/jbos/Atenuis_Philippines/metadat/all_Atenuis_sites.csv')
metadat<-read.csv('/home/jbos/Atenuis_Philippines/metadat/metadata_shoredist.csv')

In [15]:
#Read in VCF of SNPs from reads mapped to Cladocopium genome
vcf<-read.vcfR("/scratch/jbos/cladocopium/pruned_snps10.vcf")

Scanning file to determine attributes.
File attributes:
  meta lines: 699
  header_line: 700
  variant count: 763
  column count: 241
Meta line 699 read in.
All meta lines processed.
gt matrix initialized.
Character matrix gt created.
  Character matrix gt rows: 763
  Character matrix gt cols: 241
  skip: 0
  nrows: 763
  row_num: 0
Processed variant: 763
All variants processed


In [16]:
#Convert VCF to genind and create population variable from sampling locations
genind0 <- vcfR2genind(vcf)
genind0@pop <-as.factor(unlist(lapply(strsplit(rownames(genind0$tab),"_"), head, 1)))

In [17]:
#Get metadata for only sites with samples
sites0<-sites[sites$Name %in% unique(genind0@pop),]

In [18]:
#Convert genind to hierfstat to prepare for FST calculation
hstat0<-genind2hierfstat(genind0,pop=genind0@pop)

In [19]:
#Convert distances to wide form
dists0<-wide_dists(sites0)

In [20]:
#Make sf object out of points(this will be useful later)
sites_sf<-st_as_sf(sites,coords=c('LONGITUDE','LATITUDE'))
sites_sf<-st_set_crs(sites_sf,4326)
sitepoints<-st_combine(sites_sf)
sites_proj<-st_transform(sitepoints,crs=9834)

In [21]:
#Calculate FSTs between sampling sites (note that this is time consuming)
#fsts<-pairwise.WCfst(hstat0)
#write.csv(fsts,'/home/jbos/Philippines_revisions/cladocopium_fsts_all_missingness.csv',row.names=TRUE)

In [22]:
#Alternatively, read in FSTs
fsts<-as.data.frame(read_csv('/home/jbos/Philippines_revisions/cladocopium_fsts_all_missingness.csv',show_col_types = FALSE))

New names:
• `` -> `...1`


In [23]:
#Count individuals per population. Obviously this is individual corals per site, not individiual symbionts
inds_per_pop0<-as.data.frame(cbind(rownames(genind0@tab),as.character(genind0@pop)))
colnames(inds_per_pop0)<-c("Individual","Population")

In [24]:
c1<-make_counts(inds_per_pop0)
acropora_counts<-read.csv('/home/jbos/Philippines_revisions/site_counts_missingness.csv')
counts<-full_join(c1,acropora_counts)
counts<-counts[,-3]

Joining with `by = join_by(pop)`


In [25]:
#Convert FSTs to long form

fsts_long<-fst_transform(fsts,inds_per_pop0)

In [26]:
#Calculate linearized FSTs from wide form FSTs
fst_lin<-as.data.frame(linear_fsts(fsts))

In [27]:
#Select sites from Cebu only and Leyte only
Ceb_real <- sites[sites$LONGITUDE<124.25,3]
Leyte<-sites[substr(sites$Name,start=1,stop=3)=='LEY',3]

In [28]:
#Take log of distance for 2D Mantel test
log_dist<-log(dists0)

In [29]:
#Create "inds_per_pop" variable to throw out populations with single individuals that are throwing errors
inds_per_pop<-as.data.frame(cbind(rownames(genind0@tab),as.character(genind0@pop)))
colnames(inds_per_pop)<-c("Individual","Population")

In [33]:
#Throw out populations with single individuals
rownames(fsts)<-fsts$...1
fst_trim<-fst_trim_by_N(fsts,inds_per_pop,2)

In [35]:
#Do the same thing with linearized FSTs
fst_lin_trim<-as.data.frame(linear_fsts2(fst_trim))

In [36]:
#Make distance df with the same dimentions as the fst df
dists_trim<-dists0[rownames(dists0) %in% colnames(fst_trim),colnames(dists0) %in% colnames(fst_trim)]
log_dist_trim<-log(dists_trim)

In [37]:
#Make dataframes of linearized FSTs for Leyte only and Cebu only
fst_Cebu<-fst_lin[rownames(fst_lin) %in% Ceb_real, colnames(fst_lin) %in% Ceb_real]
fst_Leyte<-fst_lin[rownames(fst_lin) %in% Leyte, colnames(fst_lin) %in% Leyte]

In [38]:
#Make dataframes of linearized FSTs for Leyte only and Cebu only
fst_Cebu_trim<-fst_lin_trim[rownames(fst_lin_trim) %in% Ceb_real, colnames(fst_lin_trim) %in% Ceb_real]
fst_Leyte_trim<-fst_lin_trim[rownames(fst_lin_trim) %in% Leyte, colnames(fst_lin_trim) %in% Leyte]

In [39]:
#Make dataframes of between site distances for Leyte only and Cebu only
dists_Cebu<-dists0[rownames(dists0) %in% Ceb_real, colnames(dists0) %in% Ceb_real]
dists_Leyte<-dists0[rownames(dists0) %in% Leyte, colnames(dists0) %in% Leyte]

In [40]:
#Make dataframes of between site distances for Leyte only and Cebu only
dists_Cebu_trim<-dists_trim[rownames(dists_trim) %in% Ceb_real, colnames(dists_trim) %in% Ceb_real]
dists_Leyte_trim<-dists_trim[rownames(dists_trim) %in% Leyte, colnames(dists_trim) %in% Leyte]

In [41]:
#Reshape data for linear models and plotting
merge_trim<-fst_dist_join(fst_lin_trim,dists_trim)
merge_trim$Dist_log<-log(merge_trim$Distance_m)
merge_trim$Dist_log_km<-log(merge_trim$Distance_m/1000)
merge_Ceb_trim<-fst_dist_join(fst_Cebu_trim,dists_Cebu_trim)
merge_Ceb_trim$Dist_km<-(merge_Ceb_trim$Distance_m/1000)

Joining with `by = join_by(SiteA, SiteB)`
Joining with `by = join_by(SiteA, SiteB)`


In [42]:
#Get standard error of IBD slope and mean slope, trimmed data. Resample 1000 times.
set.seed(42)
se_trim<-summary(lm(Lin_FST ~ Dist_log,dat=merge_trim))$coefficients[2,2]
m_trim<-summary(lm(Lin_FST ~ Dist_log,dat=merge_trim))$coefficients[2,1]
ms_trim<-rnorm(1000, mean = m_trim, sd = se_trim)

In [43]:
#Get standard error of IBD slope and mean slope for Cebu. Resample 1000 times.
set.seed(42)
se_Ceb_trim<-summary(lm(Lin_FST ~ Distance_m,dat=merge_Ceb_trim))$coefficients[2,2]
m_Ceb_trim<-summary(lm(Lin_FST ~ Distance_m,dat=merge_Ceb_trim))$coefficients[2,1]
ms_Ceb_trim<-rnorm(1000, mean = m_Ceb_trim, sd = se_Ceb_trim)

In [44]:
m_trim
se_trim

m_Ceb_trim
se_Ceb_trim

[1] 0.006199499

[1] 0.001331443

[1] 1.284077e-07

[1] 2.570995e-08

In [45]:
#Check quantiles of resampled slopes, as well as % positive
m_trim
quantile(ms_trim,probs=c(0.025,0.975))
table(ms_trim>0)

[1] 0.006199499

2.5%       97.5% 
0.003526100 0.008699552


TRUE 
1000 

In [46]:
#Check quantiles of resampled slopes, as well as % positive, Cebu
m_Ceb_trim
quantile(ms_Ceb_trim,probs=c(0.025,0.975))
table(ms_Ceb_trim>0)

[1] 1.284077e-07

2.5%        97.5% 
7.678483e-08 1.766834e-07


TRUE 
1000 

In [47]:
#Read in list of individuals belonging to each Acropora taxon
taxa1<-read_csv('taxa1_inds.csv',show_col_types = FALSE)$x
taxa1<-taxa1[taxa1 %in% rownames(genind0@tab)]
taxa2<-read_csv('taxa2_inds.csv',show_col_types = FALSE)$x
taxa2<-taxa2[taxa2 %in% rownames(genind0@tab)]
taxa3<-read_csv('taxa3_inds.csv',show_col_types = FALSE)$x
taxa3<-taxa3[taxa3 %in% rownames(genind0@tab)]
taxa4<-read_csv('taxa4_inds.csv',show_col_types = FALSE)$x
taxa4<-taxa4[taxa4 %in% rownames(genind0@tab)]

New names:
• `` -> `...1`
New names:
• `` -> `...1`
New names:
• `` -> `...1`
New names:
• `` -> `...1`


In [48]:
#Separate Cladocopium belonging to samples from each cryptic taxon
clad1<-genind0[taxa1,]
clad2<-genind0[taxa2,]
clad3<-genind0[taxa3,]
clad4<-genind0[taxa4,]

In [49]:
#Make dataframes of individuals and corresponding populations for each taxon
inds_per_pop1<-as.data.frame(cbind(rownames(clad1@tab),as.character(clad1@pop)))
colnames(inds_per_pop1)<-c("Individual","Population")

inds_per_pop2<-as.data.frame(cbind(rownames(clad2@tab),as.character(clad2@pop)))
colnames(inds_per_pop2)<-c("Individual","Population")

inds_per_pop3<-as.data.frame(cbind(rownames(clad3@tab),as.character(clad3@pop)))
colnames(inds_per_pop3)<-c("Individual","Population")

inds_per_pop4<-as.data.frame(cbind(rownames(clad4@tab),as.character(clad4@pop)))
colnames(inds_per_pop4)<-c("Individual","Population")

In [50]:
#Make site counts dataframe to count individuals per taxon per site
c1<-make_counts(inds_per_pop1)
c2<-make_counts(inds_per_pop2)
c3<-make_counts(inds_per_pop3)
c4<-make_counts(inds_per_pop4)

site_counts<-full_join(c1,c2,by='pop')
site_counts<-full_join(site_counts,c3,by='pop')
site_counts<-full_join(site_counts,c4,by='pop')
colnames(site_counts)<-c('pop','taxon1','taxon2','taxon3','taxon4')
site_counts[is.na(site_counts)] <- 0
write.csv(site_counts,'site_counts_missingness_cladocopium.csv')

In [51]:
#Get metadata for only sites with samples by cryptic taxon
sites1<-sites[sites$Name %in% unique(clad1@pop),]
dists1<-wide_dists(sites1)
sites2<-sites[sites$Name %in% unique(clad2@pop),]
dists2<-wide_dists(sites2)
sites3<-sites[sites$Name %in% unique(clad3@pop),]
dists3<-wide_dists(sites3)
sites4<-sites[sites$Name %in% unique(clad4@pop),]
dists4<-wide_dists(sites4)

In [52]:
#Convert to hierfstat
hstat1<-genind2hierfstat(clad1,pop=clad1@pop)
hstat2<-genind2hierfstat(clad2,pop=clad2@pop)
hstat3<-genind2hierfstat(clad3,pop=clad3@pop)
hstat4<-genind2hierfstat(clad4,pop=clad4@pop)

In [53]:
#Calculate FSTs between sampling sites (note that this is time consuming)
fsts1<-pairwise.WCfst(hstat1)
write.csv(fsts1,'/home/jbos/Philippines_revisions/cladocopium_fsts_taxa1_missingness.csv',row.names=TRUE)

In [82]:
#Calculate FSTs between sampling sites (note that this is time consuming)
fsts<-pairwise.WCfst(hstat2)
write.csv(fsts2,'/home/jbos/Philippines_revisions/cladocopium_fsts_taxa2_missingness.csv',row.names=TRUE)

In [81]:
#Calculate FSTs between sampling sites (note that this is time consuming)
fsts4<-pairwise.WCfst(hstat3)
write.csv(fsts3,'/home/jbos/Philippines_revisions/cladocopium_fsts_taxa3_missingness.csv',row.names=TRUE)

In [80]:
#Calculate FSTs between sampling sites (note that this is time consuming)
fsts3<-pairwise.WCfst(hstat4)
write.csv(fsts4,'/home/jbos/Philippines_revisions/cladocopium_fsts_taxa4_missingness.csv',row.names=TRUE)

In [54]:
#Alternatively, read in FSTs for Cladocopium in each Acropora cryptic taxon
fsts1<-linear_fsts(read.csv('/home/jbos/Philippines_revisions/cladocopium_fsts_taxa1_missingness.csv'))
fsts2<-linear_fsts(read.csv('/home/jbos/Philippines_revisions/cladocopium_fsts_taxa2_missingness.csv'))
fsts3<-linear_fsts(read.csv('/home/jbos/Philippines_revisions/cladocopium_fsts_taxa3_missingness.csv'))
fsts4<-linear_fsts(read.csv('/home/jbos/Philippines_revisions/cladocopium_fsts_taxa4_missingness.csv'))

In [55]:
#Create matrix of FSTs only including populations where N>1
fst_trim1<-fst_trim_by_N(fsts1,inds_per_pop1,2)
fst_trim2<-fst_trim_by_N(fsts2,inds_per_pop2,2)
fst_trim3<-fst_trim_by_N(fsts3,inds_per_pop3,2)
fst_trim4<-fst_trim_by_N(fsts4,inds_per_pop4,2)

In [72]:
#Merge FSTs and geographic distances
merge1<-fst_dist_join(fst_trim1,dists1)
merge1$Dist_log<-log(merge1$Distance_m)

merge2<-fst_dist_join(fst_trim2,dists2)
merge2$Dist_log<-log(merge2$Distance_m)

merge3<-fst_dist_join(fst_trim3,dists3)
merge3$Dist_log<-log(merge3$Distance_m)

merge4<-fst_dist_join(fst_trim4,dists4)
merge4$Dist_log<-log(merge4$Distance_m)

Joining with `by = join_by(SiteA, SiteB)`
Joining with `by = join_by(SiteA, SiteB)`
Joining with `by = join_by(SiteA, SiteB)`
Joining with `by = join_by(SiteA, SiteB)`


In [83]:
nrow(fst_trim1)
nrow(fst_trim2)
nrow(fst_trim3)
nrow(fst_trim4)

[1] 22

[1] 15

[1] 9

[1] 7

In [73]:
#Slope, etc. for taxon 1
set.seed(42)
se1<-summary(lm(Lin_FST ~ Dist_log,dat=merge1))$coefficients[2,2]
m1<-summary(lm(Lin_FST ~ Dist_log,dat=merge1))$coefficients[2,1]
ms1<-rnorm(1000, mean = m1, sd = se1)
table(ms1>0)


FALSE  TRUE 
  290   710 

In [74]:
#Slope, etc. for taxon 2
set.seed(42)
se2<-summary(lm(Lin_FST ~ Dist_log,dat=merge2))$coefficients[2,2]
m2<-summary(lm(Lin_FST ~ Dist_log,dat=merge2))$coefficients[2,1]
ms2<-rnorm(1000, mean = m2, sd = se2)
table(ms2>0)


TRUE 
1000 

In [75]:
#Slope, etc. for taxon 3
set.seed(42)
se3<-summary(lm(Lin_FST ~ Dist_log,dat=merge3))$coefficients[2,2]
m3<-summary(lm(Lin_FST ~ Dist_log,dat=merge3))$coefficients[2,1]
ms3<-rnorm(1000, mean = m3, sd = se3)
table(ms3>0)


FALSE  TRUE 
  216   784 

In [76]:
#Slope, etc. for taxa 4
set.seed(42)
se4<-summary(lm(Lin_FST ~ Dist_log,dat=merge4))$coefficients[2,2]
m4<-summary(lm(Lin_FST ~ Dist_log,dat=merge4))$coefficients[2,1]
ms4<-rnorm(1000, mean = m4, sd = se4)
table(ms4>0)


FALSE  TRUE 
  428   572 

In [77]:
#Select sites on Cebu from each taxon for 1D analysis
merge1_Ceb_trim<-merge1[(merge1$SiteA %in% Ceb_real) & (merge1$SiteB %in% Ceb_real),]
merge2_Ceb_trim<-merge2[(merge2$SiteA %in% Ceb_real) & (merge2$SiteB %in% Ceb_real),]
merge3_Ceb_trim<-merge3[(merge3$SiteA %in% Ceb_real) & (merge3$SiteB %in% Ceb_real),]
merge4_Ceb_trim<-merge4[(merge4$SiteA %in% Ceb_real) & (merge4$SiteB %in% Ceb_real),]

In [78]:
#Slope, etc. for taxon 1
set.seed(42)
se1<-summary(lm(Lin_FST ~ Dist_log,dat=merge1_Ceb_trim))$coefficients[2,2]
m1<-summary(lm(Lin_FST ~ Dist_log,dat=merge1_Ceb_trim))$coefficients[2,1]
ms1<-rnorm(1000, mean = m1, sd = se1)
table(ms1>0)


FALSE  TRUE 
  132   868 

In [79]:
#Slope, etc. for taxon 2
set.seed(42)
se2<-summary(lm(Lin_FST ~ Dist_log,dat=merge2_Ceb_trim))$coefficients[2,2]
m2<-summary(lm(Lin_FST ~ Dist_log,dat=merge2_Ceb_trim))$coefficients[2,1]
ms2<-rnorm(1000, mean = m2, sd = se2)
table(ms2>0)


TRUE 
1000 

In [80]:
#Slope, etc. for taxon 3
set.seed(42)
se3<-summary(lm(Lin_FST ~ Dist_log,dat=merge3_Ceb_trim))$coefficients[2,2]
m3<-summary(lm(Lin_FST ~ Dist_log,dat=merge3_Ceb_trim))$coefficients[2,1]
ms3<-rnorm(1000, mean = m3, sd = se3)
table(ms3>0)


FALSE  TRUE 
  190   810 

In [81]:
#Slope, etc. for taxon 4
set.seed(42)
se4<-summary(lm(Lin_FST ~ Dist_log,dat=merge4_Ceb_trim))$coefficients[2,2]
m4<-summary(lm(Lin_FST ~ Dist_log,dat=merge4_Ceb_trim))$coefficients[2,1]
ms4<-rnorm(1000, mean = m4, sd = se4)
table(ms4>0)


FALSE  TRUE 
  428   572 